In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from encoder_decoder import*

# Playing around with encodings and embeddings to learn how to use them

In [2]:
# first import the little dictionary frol it_es_cognates.txt

italian_words = []
spanish_words = []

with open("it_es_cognates.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        it, es = line.split(";") # words separated by ;
        italian_words.append(it.lower())
        spanish_words.append(es.lower())

# check the number of words in the list of cognates
print(len(italian_words), "pairs")
print(italian_words[:5], spanish_words[:5])

# find all characters appearing in the list of cognates
all_chars = set()
for word in spanish_words + italian_words:
    all_chars.update(word)
all_chars = sorted(all_chars)

# add the special caracters: pad, start of string, end of string, unknown
specials = ['<pad>', '<sos>', '<eos>', '<unk>']
vocab = specials + all_chars

# build the dictionaries, just use the enumeration of vocab to assign an integer to every character
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# check the characters in the vocabulary, and its length
print("vocab: ", vocab)
print("vocab length:", len(vocab))

# test word size in the vocabulary (need to know the dimension of the words I am going to have in the model - by padding)
# I will add 2 or 3 just to be on the safe side for future additions to the dictionary, getting, say, to 20
print("Max length of spanish words: ", max(len(w) for w in spanish_words))
print("Max length of italian words: ", max(len(w) for w in italian_words))

def encode_source(word, max_len = 20):
    # input: no sos/eos needed
    ids = [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len] # use the char to int dictionary
    ids += [char_to_idx['<pad>']] * (max_len - len(ids)) # fill with '<pad>' until the prescribed length max_len
    return torch.tensor(ids)

def encode_target(word, max_len = 20):
    # output: needs sos/eos since decoder generates it step by step, they will replace two <pad> 
    ids = [char_to_idx['<sos>']] + [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len-2] + [char_to_idx['<eos>']]
    ids += [char_to_idx['<pad>']] * (max_len - len(ids))
    return torch.tensor(ids) 

def decode_source(ids):
    
    ids_numpy = ids.numpy()
    chars = []
    for i in ids_numpy:
        ch = idx_to_char[i] # use the integer to char dictionary
        if ch == '<pad>':
            break  # padding marks the end of real content
        chars.append(ch)
    return ''.join(chars)

# test
print(encode_source('castoro'))
print(decode_source(encode_source('castoro'))) 
print(decode_source(encode_source('è'))) # check unknown characters


# now test the embedding
vocab_size = len(char_to_idx)
d_model = 32 # needs to be large enough but does not have to be larger than number of characters - in fact for LLMs it is smaller than the number of tokens
embed = nn.Embedding(vocab_size, d_model, padding_idx=char_to_idx['<pad>'])

print(embed(encode_source('castoro')))

908 pairs
['acqua', 'aglio', 'aiutare', 'aiutata', 'aiutate'] ['agua', 'ajo', 'ayudar', 'ayudada', 'ayudadas']
vocab:  ['<pad>', '<sos>', '<eos>', '<unk>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', 'à', 'á', 'é', 'í', 'ñ', 'ó', 'ù', 'ú']
vocab length: 36
Max length of spanish words:  15
Max length of italian words:  13
tensor([ 6,  4, 21, 22, 17, 20, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0])
castoro
<unk>
tensor([[-0.1222, -1.6885,  1.1477,  0.8410,  0.8971, -1.0301, -0.6423,  0.1609,
         -0.1770,  0.4524,  0.7840, -0.7190,  0.6736, -0.0634, -0.7290, -0.2002,
         -1.0616,  0.7533,  0.7746,  0.7579, -1.2012, -0.6986,  1.0859,  0.2980,
         -1.5284,  0.2677, -0.9899,  0.6268,  0.2323,  0.3072, -0.2263,  0.2512],
        [ 0.0559, -0.5268, -0.8842, -0.0126, -1.1567, -0.3803,  0.2783, -1.2915,
         -1.2074, -1.3582, -1.0014,  0.3319,  1.9201, -0.4522, -0.4274,  0.1423,
   

In [5]:
# test if Encoder runs correctly

enco = Encoder(2, d_model = d_model, d_hidden = 4*d_model, dk = 8, dv = 8, h = 4)
pos_enco = PositionalEncodingModule(d_model=d_model)

encoded_word = pos_enco(enco(embed(encode_source('castoro')).unsqueeze(0)))
print(encoded_word)

#ok

tensor([[[-6.5672e-01, -1.8973e+00,  1.8160e+00,  2.1312e+00,  4.1353e-01,
           1.3181e-01, -9.7964e-01,  1.1170e+00, -1.5685e-01,  1.5387e+00,
           1.0754e+00,  5.3481e-02,  5.0206e-01,  3.2263e-01, -6.2426e-01,
           1.2019e+00, -1.0040e+00,  1.6266e+00,  1.2910e+00,  2.4039e+00,
          -1.0142e+00,  7.1464e-01,  1.1560e+00,  1.3173e+00, -1.6519e+00,
           1.0713e+00, -7.8039e-01,  1.5855e+00,  1.2932e+00,  1.2824e+00,
          -3.9179e-01,  1.1113e+00],
         [ 3.0328e-01, -5.6819e-01,  2.8452e-01,  9.7702e-01, -9.2667e-01,
           1.0923e+00,  4.2746e-01, -2.3460e-01, -4.8713e-01, -4.3775e-01,
          -7.8517e-01,  1.5577e+00,  1.5699e+00,  3.3365e-01, -3.1430e-02,
           1.1435e+00,  6.4339e-02,  2.3638e+00,  1.3116e+00,  1.4547e+00,
          -1.2129e+00,  2.4378e+00,  1.2811e+00,  1.4168e+00,  5.0631e-01,
           1.4775e+00,  1.2752e+00, -9.4075e-01, -8.6275e-01,  2.5104e+00,
           1.0011e+00, -3.0900e-01],
         [ 1.6483e+00, -2.